# 1 · Model

Notebook 0's model is three modules glued together with `nn.Sequential`. That is fine until it has to change:

- The channel count 16 is written in all three parts; widening the encoder means editing the other two to match.
- Trying another encoder means rewriting the glue.

This notebook gives each part a declared interface so the parts can change one at a time. We also register our own model parts, load a small pretrained model from the built-in catalogue, add augmentation, and compare the runs in TensorBoard. The task and the data stay the same.

The same data, opened the same way. If notebook 0 already fetched it, the download reuses those files.

In [1]:
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from torch import nn
from torch.utils.data import DataLoader, Dataset

import geosave_engine as gs

root = Path(snapshot_download("fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"))


class Tiles(Dataset):
    def __init__(self, split):
        self.images = sorted((root / split / "images").glob("*.tif"))
        self.labels = sorted((root / split / "labels").glob("*.tif"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = gs.open_raster(self.images[index]).isel(time=0)
        label = gs.open_raster(self.labels[index])["label"].isel(time=0)
        return image.gs.to_tensor(dtype="float32") / 10_000, label.gs.to_tensor(dtype="int64")


train = Tiles("train")
val = Tiles("val")
train_loader = DataLoader(train, batch_size=8, shuffle=True)
val_loader = DataLoader(val, batch_size=8)

Fetching 603 files:   0%|          | 0/603 [00:00<?, ?it/s]

## The same three parts

These are still ordinary PyTorch modules: an encoder, a decoder, and a head. We add declarations to let GeoSave connect them and build them by name; their layers and direct calls work as before.

In [2]:
from geosave_engine.model import ModelChain, Published, build_model, chain_step
from geosave_engine.model.registry import register_model


@register_model("encoder", "tiny")
class Encoder(nn.Module):
    channels: Published[int]

    def __init__(self, channels: int = 16):
        super().__init__()
        self.channels = channels
        self.net = nn.Sequential(nn.Conv2d(4, channels, 3, stride=2, padding=1), nn.ReLU())

    @chain_step()
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        features = self.net(image)
        return features


@register_model("decoder", "upsample")
class Decoder(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU())

    @chain_step()
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        feature_map = self.net(features)
        return feature_map


@register_model("head", "pixel")
class Head(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Conv2d(channels, 10, 1)

    @chain_step(head=True)
    def forward(self, feature_map: torch.Tensor) -> torch.Tensor:
        return self.net(feature_map)

In [3]:
model = nn.Sequential(Encoder(), Decoder(channels=16), Head(channels=16))
model

Sequential(
  (0): Encoder(
    (net): Sequential(
      (0): Conv2d(4, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
    )
  )
  (1): Decoder(
    (net): Sequential(
      (0): Upsample(scale_factor=2.0, mode='nearest')
      (1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (2): ReLU()
    )
  )
  (2): Head(
    (net): Conv2d(16, 10, kernel_size=(1, 1), stride=(1, 1))
  )
)

## Connect the parts by what they read and return

`nn.Sequential` works well when each `forward` takes the previous part's result. Other models need richer interfaces: an encoder might return several feature maps, or a decoder might need both encoded features and the original image. Without a shared interface, swapping either part can mean writing new glue for that encoder–decoder pair.

`chain_step` lets each part declare the named values it needs and produces. When an encoder and several decoders share that interface, the encoder can support all of them without a separate adapter for each combination. Each part describes its own interface once.

`ModelChain` connects these inputs and outputs into a dependency graph. Here, the encoder produces `features`, the decoder turns them into `feature_map`, and the head returns the final scores. Displaying the model shows that data flow above its layers.

In [4]:
model = ModelChain(
    encoder=Encoder(),
    decoder=Decoder(channels=16),
    head=Head(channels=16),
)
model

```mermaid
flowchart LR
    input_image(["image: Tensor"])
    encoder["encoder: Encoder.forward"]
    decoder["decoder: Decoder.forward"]
    head["head: Head.forward"]
    result_head(["Tensor"])
    input_image -->|"image"| encoder
    encoder -->|"features: Tensor"| decoder
    decoder -->|"feature_map: Tensor"| head
    head --> result_head
```

## Swap the decoder

This decoder resizes the features to the original image's dimensions. Its `forward` therefore needs two inputs: `features` from the encoder and `image` from the caller. Swap it into the same chain:

In [5]:
class ResizeDecoder(nn.Module):
    @chain_step()
    def forward(self, features: torch.Tensor, image: torch.Tensor) -> torch.Tensor:
        feature_map = nn.functional.interpolate(
            features, size=image.shape[-2:], mode="bilinear", align_corners=False
        )
        return feature_map


model = ModelChain(
    encoder=Encoder(),
    decoder=ResizeDecoder(),
    head=Head(channels=16),
)
model

```mermaid
flowchart LR
    input_image(["image: Tensor"])
    encoder["encoder: Encoder.forward"]
    decoder["decoder: ResizeDecoder.forward"]
    head["head: Head.forward"]
    result_head(["Tensor"])
    input_image -->|"image"| encoder
    encoder -->|"features: Tensor"| decoder
    input_image -->|"image"| decoder
    decoder -->|"feature_map: Tensor"| head
    head --> result_head
```

The encoder and head need no changes. `ModelChain` keeps the caller's inputs and named outputs from earlier steps available throughout the forward call. Each step receives only the values it asks for, so this decoder can still receive `image` after the encoder has produced `features`. The caller still uses `model(image=image)`.

This works with normal PyTorch training. Here, the decoder reads only the image's dimensions; gradients flow through the resized features back to the encoder. Passing a tensor to another step preserves its gradient connection. We do not detach the features, because we want the encoder to learn from the final loss.

## Change the width in one place

We still had to repeat the channel count when constructing the parts. The encoder's class attribute `channels: Published[int]` makes its width available to parts built after it. The decoder and head already take `channels`, so they can receive that value without knowing which encoder supplied it.

`build_model` constructs the parts in order and passes along published values. Give it the classes and it builds the same model using the encoder's default width:

In [6]:
model = build_model({"encoder": Encoder, "decoder": Decoder, "head": Head})
model

```mermaid
flowchart LR
    input_image(["image: Tensor"])
    encoder["encoder: Encoder.forward"]
    decoder["decoder: Decoder.forward"]
    head["head: Head.forward"]
    result_head(["Tensor"])
    input_image -->|"image"| encoder
    encoder -->|"features: Tensor"| decoder
    decoder -->|"feature_map: Tensor"| head
    head --> result_head
```

Now widening the encoder is one change. Give it a module built with 32 channels; the decoder and head follow on their own. The model's displayed layers show the new widths.

In [7]:
model = build_model({"encoder": Encoder(channels=32), "decoder": Decoder, "head": Head})
model

```mermaid
flowchart LR
    input_image(["image: Tensor"])
    encoder["encoder: Encoder.forward"]
    decoder["decoder: Decoder.forward"]
    head["head: Head.forward"]
    result_head(["Tensor"])
    input_image -->|"image"| encoder
    encoder -->|"features: Tensor"| decoder
    decoder -->|"feature_map: Tensor"| head
    head --> result_head
```

## Build your own registered model

The `@register_model` decorators on our classes give each implementation a name within its stage: `tiny`, `upsample`, and `pixel`. We can select these custom parts by name just as we would select a built-in model.

Here is the same wider model as a construction recipe. Only the encoder needs `channels` in `init_args`; the decoder and head still receive its published width. Because this recipe is plain data, it can also live in a YAML or JSON config.

In [8]:
model = build_model({
    "encoder": {"name": "tiny", "init_args": {"channels": 32}},
    "decoder": {"name": "upsample"},
    "head": {"name": "pixel"},
})
model

```mermaid
flowchart LR
    input_image(["image: Tensor"])
    encoder["encoder: Encoder.forward"]
    decoder["decoder: Decoder.forward"]
    head["head: Head.forward"]
    result_head(["Tensor"])
    input_image -->|"image"| encoder
    encoder -->|"features: Tensor"| decoder
    decoder -->|"feature_map: Tensor"| head
    head --> result_head
```

## Load a built-in model

Our registered classes join the same catalogue as GeoSave's built-in models. It includes DINOv3, Prithvi and Clay encoders, DPT and U-Net decoders, and heads for segmentation, classification, regression and detection. We can see the available names by stage:


In [9]:
from geosave_engine.model.registry import list_models

list_models()

{'encoder': ['TINY', 'CLAY', 'DINOV3', 'PRITHVI', 'PRITHVI_TL'],
 'decoder': ['UPSAMPLE', 'DPT', 'UNET'],
 'head': ['PIXEL', 'CLASSIFICATION', 'REGRESSION', 'SEGMENTATION'],
 'model': ['IBM_GRANITE_BIOMASS']}

Let's load [DINOv3](https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m) ViT-S/16, the smallest built-in DINOv3 encoder, with about 21 million parameters. `pretrained` is on by default, so the weights download on the first run and come from the cache afterward.

We pair it with a U-Net decoder and a new segmentation head for our ten classes. The label files carry their class names in a legend; we take every class except 255, the unlabelled pixels. The encoder publishes its feature widths and strides; the decoder and head receive the values they need through `build_model`, just as our custom parts did. Only the encoder starts with pretrained weights; the decoder and head need training for the task.

In [10]:
legend = gs.open_raster(train.labels[0])["label"].gs.legend
classes = [name for value, name in legend.classes.items() if value != 255]

pretrained_model = build_model({
    "encoder": {
        "name": "dinov3",
        "init_args": {
            "model_name": "vit_small_patch16_dinov3.lvd1689m",
            "in_channels": 4,
            "dynamic_img_pad": True,
        },
    },
    "decoder": {"name": "unet"},
    "head": {
        "name": "segmentation",
        "init_args": {"classes": classes},
    },
})
pretrained_model

2026-10-12 11:00:55,682 - INFO - Loading pretrained weights from Hugging Face hub (timm/vit_small_patch16_dinov3.lvd1689m)


2026-10-12 11:00:56,035 - INFO - HTTP Request: HEAD https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m/resolve/main/model.safetensors "HTTP/1.1 302 Found"


2026-10-12 11:00:56,047 - INFO - [timm/vit_small_patch16_dinov3.lvd1689m] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


2026-10-12 11:00:56,190 - INFO - Converted input conv patch_embed.proj pretrained weights from 3 to 4 channel(s)


```mermaid
flowchart LR
    input_image(["image: Tensor"])
    encoder["encoder: DINOv3.forward_pyramid"]
    decoder["decoder: Unet.forward"]
    head["head: SegmentationHead.forward_logits"]
    result_head(["Tensor"])
    input_image -->|"image"| encoder
    encoder -->|"prefix_tokens: list[torch.Tensor | None]"| result_prefix_tokens(["prefix_tokens"])
    encoder -->|"pyramid: list[torch.Tensor]"| decoder
    decoder -->|"feature_map: Tensor"| head
    head --> result_head
```

DINOv3 learned from photos, so its first layer has one learned kernel for each of red, green and blue. With `in_channels=4` the fourth band reuses the red kernel, and the first three keep theirs only if our bands arrive as red, green and blue in that order. Our tiles hold blue first, so training this encoder would start by reordering the bands by name: `image[["B04", "B03", "B02", "B08"]]`.

Our tiles are 510 pixels wide, which is not a multiple of the encoder's 16-pixel patches; `dynamic_img_pad` pads each tile to the next multiple and the prediction comes back at 510.

We keep training the custom `model` below. Notebook 2 prepares better inputs, and notebook 3 fetches imagery with dates and places, which encoders such as Prithvi also read.

## Augmentation

200 tiles is little data for a segmentation model. Flipping each batch at random shows the model a different view every step. [Kornia](https://kornia.readthedocs.io) augments whole batches; its `AugmentationSequential` applies the same flip to the image and its labels so they stay matched, and `data_keys` says which input is which. `keepdim=True` returns each tensor in the shape it came in, so the labels stay `(batch, height, width)`.

The rest is the same Lightning training setup as notebook 0, now using our registered model.

In [11]:
import kornia.augmentation as K
import lightning as L
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.augment = K.AugmentationSequential(
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["image", "mask"],
            keepdim=True,
        )
        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(num_classes=10, ignore_index=255),
                "accuracy": MulticlassAccuracy(num_classes=10, ignore_index=255, average="micro"),
            },
            prefix="val_",
        )
        self.criterion = nn.CrossEntropyLoss(ignore_index=255)

    def forward(self, image):
        return self.model(image=image)

    def training_step(self, batch, batch_index):
        image, label = self.augment(*batch)
        loss = self.criterion(self(image), label)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch
        logits = self(image)
        self.log("val_loss", self.criterion(logits, label))
        self.val_metrics(logits, label)
        self.log_dict(self.val_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(model)
module

Segmenter(
  (model): inputs: image: Tensor

  data flow:
    encoder: Encoder.forward(image: Tensor) -> {features: Tensor}
    decoder: Decoder.forward(features: Tensor) -> {feature_map: Tensor}
    head: Head.forward(feature_map: Tensor) -> Tensor

  ModelChain(
    (encoder): Encoder(
      (net): Sequential(
        (0): Conv2d(4, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): ReLU()
      )
    )
    (decoder): Decoder(
      (net): Sequential(
        (0): Upsample(scale_factor=2.0, mode='nearest')
        (1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (2): ReLU()
      )
    )
    (head): Head(
      (net): Conv2d(32, 10, kernel_size=(1, 1), stride=(1, 1))
    )
  )
  (augment): AugmentationSequential(
    (RandomHorizontalFlip_0): RandomHorizontalFlip(p=0.5, p_batch=1.0, same_on_batch=False)
    (RandomVerticalFlip_1): RandomVerticalFlip(p=0.5, p_batch=1.0, same_on_batch=False)
  )
  (val_metrics): MetricCollection(
    (accu

In [12]:
import warnings

from lightning.pytorch.loggers import TensorBoardLogger

# The tiles are small and few; Lightning's hint to add loader workers is noise here.
warnings.filterwarnings("ignore", ".*does not have many workers.*")

L.seed_everything(0)
trainer = L.Trainer(
    max_epochs=6,
    limit_train_batches=6,
    limit_val_batches=2,
    log_every_n_steps=1,
    logger=TensorBoardLogger("lightning_logs", name="", version="model"),
)
trainer.fit(module, train_loader, val_loader)

Seed set to 0


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


...

Trainable params: 10.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 19                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/uwu/Projects/geosave-engine/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/uwu/Projects/geosave-engine/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

`Trainer.fit` stopped: `max_epochs=6` reached.


## Compare the runs

Notebook 0 logged to `lightning_logs/basics/` and this run to `lightning_logs/model/`, both for the same few batches. TensorBoard draws them on the same axes, so we can compare validation accuracy and mIoU run by run.

In [13]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

## What happened

The model's parts can now be resized or swapped one at a time, and our own implementations can be selected by name. The same recipe also loaded a pretrained encoder with a built-in decoder and head. For our training run, we widened the custom encoder and added flips; the curves show what that run measured.

The model we trained still reads four numbers per pixel without knowing what they are. The next notebook looks at what the satellite recorded and what each picture knows about itself.

| Next | Question |
| --- | --- |
| 2 · Inputs | What do the four numbers mean, and how do I give the model better inputs? |